# W9-D3 概念实验：Deployment 为什么独立于 Release？

配套阅读：同名 `.md`。这里不复述阅读材料，而是用小规模、可运行的模型检验其中的架构约束。

## 实验问题

**问题 1：同一个 Release 能否在两个环境形成不同、可重放的执行闭包？**

Release 是可移植制品；DeploymentRevision 把环境绑定、ABI、知识和策略一起冻结。

In [ ]:
from dataclasses import dataclass, field, replace
from hashlib import sha256
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
rng = np.random.default_rng(202608)
def canonical_digest(payload):
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return "sha256:" + sha256(canonical.encode()).hexdigest()
@dataclass(frozen=True)
class DeploymentRevision:
    revision_id: str
    skill_release_digest: str
    environment: str
    binding_manifest_digest: str
    knowledge_digest: str
    policy_digest: str
    runtime_abi: str
    evaluation_only: bool = False
    def digest(self):
        return canonical_digest({k: v for k, v in self.__dict__.items() if k not in {"revision_id", "evaluation_only"}})

prod = DeploymentRevision("rev-prod-1", "sha256:release1", "prod", "sha256:binding-sh", "sha256:kb-sh", "sha256:policy-strict", "abi-2")
staging = replace(prod, revision_id="rev-stage-1", environment="staging", binding_manifest_digest="sha256:binding-stage")
print("同一 release：", prod.skill_release_digest == staging.skill_release_digest)
print("生产闭包：", prod.digest()[:25] + "...")
print("预发闭包：", staging.digest()[:25] + "...")
assert prod.digest() != staging.digest()

## 实验问题

**问题 2：为什么“运行时读取 latest 配置”破坏可复现性？**

同一 release 在两次调用间外部知识指针变化；闭包模式把运行时结果绑定到 revision 自己的 digest。

In [ ]:
latest_knowledge = {"current": "sha256:kb-a"}
def run_with_latest(release): return (release, latest_knowledge["current"])
def run_with_closure(revision): return (revision.skill_release_digest, revision.knowledge_digest)

first = run_with_latest(prod.skill_release_digest)
latest_knowledge["current"] = "sha256:kb-b"
second = run_with_latest(prod.skill_release_digest)
frozen_first, frozen_second = run_with_closure(prod), run_with_closure(prod)
print("latest 模式：", first, "->", second)
print("闭包模式：", frozen_first, "->", frozen_second)
assert first != second and frozen_first == frozen_second

## 实验问题

**问题 3：为什么回滚应是前向物化，而不是修改历史 Revision？**

从历史闭包新建 revision；历史记录保持可审计，也允许与当前版本并存。

In [ ]:
rev2 = replace(prod, revision_id="rev-prod-2", knowledge_digest="sha256:kb-new")
rollback = replace(prod, revision_id="rev-prod-3-rollback")
history = [prod, rev2, rollback]
for r in history: print(r.revision_id, r.knowledge_digest)
assert len({r.revision_id for r in history}) == 3
assert rollback.digest() == prod.digest()
print("回滚内容可相同，但审计事件和 revision 身份是新的。")

## 实验问题

**问题 4：完整闭包如何使环境差异显性化？**

画出三个环境的依赖差异数；Release 不变，差异集中在 DeploymentRevision。

In [ ]:
revisions = [prod, staging, replace(prod, revision_id="rev-dr-1", environment="dr", binding_manifest_digest="sha256:binding-dr", runtime_abi="abi-3")]
base_fields = prod.__dict__
diffs = [sum(getattr(r, f) != base_fields[f] for f in base_fields if f != "revision_id") for r in revisions]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar([r.environment for r in revisions], diffs, color=["#1b9e77", "#7570b3", "#d95f02"])
ax.set_ylabel("相对 prod 的闭包字段差异数"); ax.set_title("环境差异在 Revision 中显式保存，而非污染 Release")
plt.tight_layout(); plt.show()
print("Release digest 在三环境相同：", len({r.skill_release_digest for r in revisions}) == 1)